<a href="https://colab.research.google.com/github/Marwan77770/Marwan_FlyRank/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Marwan77770/Marwan_FlyRank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

I frame my lane as a ranking task.

The goal is to rank content pages by their priority for human review, rather than making a simple yes/no decision for each page. The ranking can combine multiple observed search, content, and engagement signals to identify which pages should be reviewed first.

This fits the Refresh / Content Opportunity Scoring lane because the practical output is a prioritized review queue. A content or search team can start with the highest-ranked pages and then decide whether each page should be refreshed, expanded, protected, pruned, or monitored.

This is decision support: the ranking helps allocate limited review capacity, but it does not automatically determine the final content action.

In [ ]:
print("ML task type: Ranking")
print("Unit of analysis: Content page")
print("Output: Prioritized review queue")

ML task type: Ranking
Unit of analysis: Content page
Output: Prioritized review queue


## 2. Target or proxy

For this ranking task, I would use is_declining_label as a proxy target.

The label is derived from the observed trend_direction field: a page is labeled 1 when its observed trend direction is "down", and 0 otherwise.

This is a defined proxy based on the current observation window, not a confirmed future outcome. It provides a measurable starting point for identifying pages associated with declining performance.

The proxy can help the model learn patterns in the available search, content, and engagement signals and use them to prioritize pages for human review. However, I would not claim that the label represents guaranteed future decline or that it proves why a page is declining.

In [ ]:
import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 30000
Columns: 44


In [ ]:
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Target column:", "is_declining_label")
print("Target type: Proxy")
print("Label rule: trend_direction == 'down'")
print("Positive labels:", df["is_declining_label"].sum())


Target column: is_declining_label
Target type: Proxy
Label rule: trend_direction == 'down'
Positive labels: 16262


## 3. Success metric

My primary success metric is Precision@50.

This measures the proportion of the top 50 ranked pages that have the positive proxy label for declining performance.

I chose Precision@50 because the practical use case involves limited human review capacity. The goal is not simply to classify every page correctly, but to make the first pages in the review queue as useful and relevant as possible.

A higher Precision@50 means that a larger share of the pages prioritized for review match the observed declining-performance proxy. I would treat this metric as a measure of prioritization quality, not as proof that refreshing a page will improve its future performance.

In [ ]:
print("Success metric: Precision@50")
print("Review capacity: 50 pages")
print("Higher Precision@50 means a more useful top-50 review queue.")


Success metric: Precision@50
Review capacity: 50 pages
Higher Precision@50 means a more useful top-50 review queue.


## 4. The unit of analysis, as a real dataframe

The unit of analysis is one content page.

Each row in the dataframe represents one content page and contains its observed search, content, engagement, and trend-related signals. The page is the entity being ranked for review priority.

The target proxy is_declining_label is also defined at the page level, so the input signals and the target are aligned to the same unit of analysis.

In [ ]:
print("One row represents: one content page")
print("Number of rows:", len(df))

display(df.head())

One row represents: one content page
Number of rows: 30000


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1


In [ ]:
print("Columns:")
print(df.columns.tolist())

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label']


## 5. Why ML beats a fixed rule here

A fixed rule could be useful as a simple baseline, for example, prioritizing every page whose observed trend direction is "down". However, this rule considers only one signal and treats all declining pages in the same way.

The dataset contains multiple observed search, content, and engagement signals. Their combination may provide more useful information for prioritizing pages. For example, a declining page with high impressions may deserve different review priority from a declining page with very low visibility.

A machine learning approach can learn patterns from multiple signals and produce a ranked priority score instead of relying on a single manually chosen threshold.

However, ML is not automatically better than a fixed rule. I would compare the ML approach against a simple rule-based baseline using the same validation setup and Precision@50. The ML approach would only be useful if it provides a meaningful improvement in prioritizing pages for human review.

In [ ]:
print("Baseline rule: prioritize pages where trend_direction == 'down'")
print("ML approach: learn from multiple observed signals")
print("Comparison metric: Precision@50")
print("Final output: ranked pages for human review")

Baseline rule: prioritize pages where trend_direction == 'down'
ML approach: learn from multiple observed signals
Comparison metric: Precision@50
Final output: ranked pages for human review


## Self-check

Before you submit, confirm each line honestly:

- Every section above is filled — markdown thinking AND the code that backs it
-  The notebook runs top to bottom with no errors (Runtime → Run all)
-  No client names, URLs, or private queries anywhere
-  My claims use careful words: observed, measured, directional, decision-support
-  Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.